RQ2 -----------------------------------------------

XGBoost

Facemask wearing covid model

Dataset with state and covid 7 day rolling cases and deaths are considered here for the analysis.

In [2]:
# import libraries
import pandas as pd

from sklearn.model_selection import StratifiedKFold  
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import joblib

state_train_facemask = pd.read_csv("state_train_facemask.csv")
state_test_facemask = pd.read_csv("state_test_facemask.csv")

state_train_facemask.columns

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'state',
       'household_size', 'Wellbeing', 'Perceived Severity',
       'Perceived Susceptibility', 'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'employment_status_Not working',
       'employment_status_Part time employment', 'employment_status_Retired',
       'employment_status_Unemployed',
       'Confidence in Goverment's response_A lot of confidence',
       'Confidence in Goverment

In [3]:
states = sorted(state_train_facemask["state"].unique()) # to get 8 states

# remove the target variables and identifiers
drop_cols = ['RecordNo', 'Date', 'state', 'face_mask_scale', 'face_mask_binary','general_protective_behavior_scale',
                'general_protective_behavior_binary','mandate_start_date']

cv = StratifiedKFold( # 5 fold cross validation
        n_splits=5,
        shuffle=True,
        random_state=42
)

  
cv_xgb = Pipeline([('ros', RandomOverSampler(random_state=42)),
                   ('xgb', XGBClassifier(random_state=42,
                                         eval_metric='logloss',
                                         n_jobs=-1))
    ])  


params = {
        'xgb__n_estimators': [100, 250],
        'xgb__learning_rate': [0.05, 0.1],  # 0.3
        'xgb__min_child_weight': [1, 5, 7],                
        'xgb__max_depth': [3,5,7]
    }

state_results = []

parameter_results = []

for state in states: # 8 RF state models are created trained 

    train_state = state_train_facemask[state_train_facemask["state"] == state]

    test_state = state_test_facemask[state_test_facemask["state"] == state]

    print(f"\nProcessing {state}")
    print(f"Training: {len(train_state)}")
    print(f"Testing: {len(test_state)}")


    # predictors
    x_train = train_state.drop(columns=drop_cols)
    x_test = test_state.drop(columns=drop_cols)

    x_train = x_train.astype(float)  # converting boolean to float 
    x_test = x_test.astype(float)

    # print(x_train.columns)
    # print(x_train.shape)

    # target variable 

    y_train = train_state["face_mask_binary"]
    y_test = test_state["face_mask_binary"]


    # XGBoost model ---------------------------------------------------------


    grid = GridSearchCV( # tunes parameters
            cv_xgb,
            params,
            cv=cv,
            scoring={  # evaluates all metrics
            'roc_auc': 'roc_auc',
            'accuracy': 'accuracy',
            'f1': 'f1'},
            refit='roc_auc',  # choose the best model
            return_train_score=False,

            n_jobs=-1 # use all available CPU cores for parallel processing
        )

    grid.fit(x_train, y_train)

    best_xgb = grid.best_estimator_   # best model

    best_parameters = grid.best_params_   # best hyper parameters

    joblib.dump(best_parameters, f"RQ2_facemask_{state}_XGBoost_bestParameters_rolling.pkl")

    parameter_results.append({"State": state, **best_parameters})



    results_xgb = pd.DataFrame(grid.cv_results_)

    pred = best_xgb.predict(x_test)
    prob = best_xgb.predict_proba(x_test)[:,1]


    accuracy = round(accuracy_score(y_test, pred),4)
    roc_auc = round(roc_auc_score(y_test, prob),4)
    f1 = round(f1_score(y_test, pred),4)


    print("Accuracy:", accuracy)
    print("ROC AUC:", roc_auc)
    print("F1:", f1)


    state_results.append({
        "State": state,
        "Accuracy": accuracy,
        "ROC AUC": roc_auc,
        "F1": f1
    })

    joblib.dump(best_xgb, f"RQ2_facemask_{state}_XGBoost_rolling.pkl") # best model 
    results_xgb.to_csv(f"RQ2_facemask_{state}_XGBoost_rolling_results.csv", index=False)



state_results = pd.DataFrame(state_results)
print(state_results)
state_results.to_csv("RQ2_facemask_XGBoost_state_results.csv", index=False)



parameter_results = pd.DataFrame(parameter_results)
parameter_results.to_csv("RQ2_facemask_XGBoost_best_parameters.csv", index=False)


Processing Australian Capital Territory
Training: 491
Testing: 137
Accuracy: 0.8686
ROC AUC: 0.9054
F1: 0.8525

Processing New South Wales
Training: 9682
Testing: 2460
Accuracy: 0.813
ROC AUC: 0.889
F1: 0.8378

Processing Northern Territory
Training: 221
Testing: 50
Accuracy: 0.76
ROC AUC: 0.7902
F1: 0.4545

Processing Queensland
Training: 6511
Testing: 1621
Accuracy: 0.8316
ROC AUC: 0.9001
F1: 0.7924

Processing South Australia
Training: 3101
Testing: 723
Accuracy: 0.9101
ROC AUC: 0.9607
F1: 0.8973

Processing Tasmania
Training: 670
Testing: 153
Accuracy: 0.9216
ROC AUC: 0.9251
F1: 0.7

Processing Victoria
Training: 8213
Testing: 2043
Accuracy: 0.8331
ROC AUC: 0.8531
F1: 0.8895

Processing Western Australia
Training: 3023
Testing: 791
Accuracy: 0.847
ROC AUC: 0.9104
F1: 0.7281
                          State  Accuracy  ROC AUC      F1
0  Australian Capital Territory    0.8686   0.9054  0.8525
1               New South Wales    0.8130   0.8890  0.8378
2            Northern Territory  

In [ ]:
train_prob = best_xgb.predict_proba(x_train)[:,1]

print("Train ROC:",roc_auc_score(y_train, train_prob))

print("Test ROC:",roc_auc_score(y_test, prob))

Train ROC: 0.973512579855027
Test ROC: 0.9104369175268063
